# DS Notebook Agent — Interactive Demo

**Prerequisites**: agent server must be running (`bash scripts/start.sh` in a separate terminal).

Run the setup cell once, then use `ask("your query")` in any cell.

In [ ]:
# ── Setup — run once ────────────────────────────────────────────────────────
import httpx
import textwrap
from IPython.display import display, Markdown, Code

API = "http://127.0.0.1:8000"
_client = httpx.Client(timeout=300)
_sid = None

def _ensure_session():
    global _sid
    if _sid is None:
        _sid = _client.post(f"{API}/sessions", json={}).raise_for_status().json()["session_id"]
        print(f"Session: {_sid}")

def seed(source: str, stdout: str = "", success: bool = True):
    """Push a pre-executed cell into the agent's context."""
    _ensure_session()
    _client.post(f"{API}/sessions/{_sid}/cells",
                 json={"source": source, "stdout": stdout, "stderr": "", "success": success}
                ).raise_for_status()
    print("✓ cell seeded")

def ask(query: str, execute: bool = False):
    """Send a natural-language query to the agent and display the result.

    execute=False (default): returns generated code as a cell for you to run.
    execute=True: agent runs the code in a sandbox and shows output.
    """
    _ensure_session()
    resp = _client.post(f"{API}/sessions/{_sid}/query", json={"query": query, "execute": execute}).raise_for_status().json()
    status = resp["status"]
    code   = resp.get("final_code", "")
    stdout = resp.get("stdout", "")
    err    = resp.get("agent_error", "")
    debug  = resp.get("debug_attempts", 0)

    if status == "done":
        badge = f"✅ Done" + (f"  _(debug attempts: {debug})_" if debug else "")
        display(Markdown(badge))
        if code:
            display(Markdown(f"```python\n{code}\n```"))
        if stdout:
            display(Markdown(f"**stdout**\n```\n{stdout.strip()}\n```"))
    else:
        display(Markdown(f"❌ Failed after {debug} debug attempts"))
        if err:
            display(Markdown(f"```\n{err}\n```"))

def reset():
    """Start a fresh session."""
    global _sid
    if _sid:
        _client.delete(f"{API}/sessions/{_sid}")
    _sid = None
    _ensure_session()

print("✓ Agent helper loaded. Use ask('...') to query.")

## Option 1 — no data (agent loads it)

In [ ]:
ask("I want to build a classification model to predict Titanic survival")

## Option 2 — seed context first, then query

In [ ]:
reset()   # fresh session

seed(
    source="import pandas as pd\nimport seaborn as sns\ndf = sns.load_dataset('titanic')\nprint(df.shape)",
    stdout="(891, 15)",
)

In [ ]:
ask("Show survival rate by passenger class as a bar chart")

In [ ]:
ask("Train a random forest and print accuracy + classification report")

## Option 3 — use local CSV

In [ ]:
reset()
seed(
    source="import pandas as pd\ndf = pd.read_csv('../data/churn.csv')\nprint(df.shape)",
    stdout="(300, 11)",
)

In [ ]:
ask("Plot churn rate by contract type")

In [ ]:
# Free-form — type any query here
ask("")